# Comparison of Greenhouse Gas Emissions for Selected Reservoirs in Myanmar and the United Kingdom

## Overview
This analysis compares GHG emissions estimated using the G-res Tool and the RE-Emission model for four selected reservoirs located in Myanmar and the United Kingdom.

## Credits

* **Analysis:** Christopher Barry, UK Centre for Ecology & Hydrology (UKCEH), Bangor, United Kingdom
* **Code & Visualization:** T. Janus — replication of Barry’s original plots using Seaborn and Matplotlib

In [ ]:
# Common code for both plotting scripts
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import ScalarFormatter
from pathlib import Path

reservoir_map = {
    "BE": "Black Esk",
    "Alaw": "Llyn Alaw",
    "MY": "Myitsone",
    "SG": "Shwegyin"
}

# --- Style setup ---
sns.set_theme(style="white", context="notebook", font_scale=1.1)
colors = ["#4E79A7", "#F28E2B"]  # muted blue and orange
#colors = ["#1B9E77", "#D95F02"]  # teal and warm brown/orange
#colors = ["#8DA0CB", "#FC8D62"]  # pastel blue and pastel orange
#colors = ["#2C7BB6", "#D7191C"]  # deep blue and red

colors_profile = [
    "#4E79A7",  # muted blue
    "#F28E2B",  # muted orange
    "#59A14F",  # muted green
    "#E15759"   # muted red
]

# --- Output directory ---
output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)

## Total (average) emissions / areal emissions

In [ ]:
# --- Load and preprocess data ---
df = pd.read_csv("total_emissions_comparison.csv", sep=";")
df.drop(df.columns[0], axis=1, inplace=True)

df["Reservoir_full_name"] = df["Reservoir"].map(reservoir_map)

# Split CO₂ and CH₄ data
df_co2 = df[df["Variable"].str.startswith("co2_")].copy()
df_ch4 = df[df["Variable"].str.startswith("ch4_")].copy()

df_co2["Pathway"] = df_co2["Variable"].str.replace("co2_", "", regex=False)
df_ch4["Pathway"] = df_ch4["Variable"].str.replace("ch4_", "", regex=False)

def plot_gas(df_gas, gas_label, save=True):
    """Plot CO₂ or CH₄ emissions with wrapped y-label, larger fonts, and properly positioned legend."""
    reservoirs = df_gas["Reservoir_full_name"].unique()
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    legend_handles = None
    legend_labels = None

    for ax, reservoir in zip(axes, reservoirs):
        subset = df_gas[df_gas["Reservoir_full_name"] == reservoir]
        melt_df = subset.melt(
            id_vars=["Pathway"], 
            value_vars=["Value_reemission", "Value_G-res-Tool"],
            var_name="Model",
            value_name="Value"
        )

        model_labels = {
            "Value_reemission": "RE-Emission",
            "Value_G-res-Tool": "G-res Tool"
        }
        melt_df["Model"] = melt_df["Model"].map(model_labels)

        sns.barplot(
            data=melt_df,
            x="Pathway",
            y="Value",
            hue="Model",
            palette=colors,
            ax=ax,
            edgecolor="k",
            linewidth=0.5,
            alpha=0.7
        )

        # --- Titles & labels ---
        ax.set_title(reservoir, fontsize=14, fontweight="bold", pad=8)
        ax.set_xlabel("", fontsize=12)
        # y-label with two lines (main label + unit)
        ax.set_ylabel(f"{gas_label} flux\n(gCO$_{{2e}}$ m$^{{-2}}$ yr$^{{-1}}$)", fontsize=13)
        #ax.set_ylabel(rf"{gas_label} flux\n(gCO$_{{2e}}$ m$^{{-2}}$ yr$^{{-1}}$)", fontsize=12)
        ax.tick_params(axis='x', rotation=30, labelsize=12)
        ax.tick_params(axis='y', labelsize=12)

        # Axis lines and grid
        for spine in ax.spines.values():
            spine.set_color("k")
            spine.set_linewidth(1.0)
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.grid(True, axis='y', linestyle='-', linewidth=0.35, alpha=0.8, color='0.7')

        # Capture handles and labels for shared legend
        if legend_handles is None:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()  # Remove per-panel legend

    # --- Shared legend below suptitle ---
    fig.suptitle(f"{gas_label} GHG components: RE-Emission vs. G-res Tool", fontsize=16, fontweight="bold", y=0.98)
    fig.legend(
        legend_handles, legend_labels,
        loc="upper center", bbox_to_anchor=(0.5, 0.95),
        ncol=2,
        frameon=False, fontsize=12,
        handlelength=1.8, handleheight=1.5
    )

    # --- Adjust spacing ---
    plt.subplots_adjust(hspace=0.9, wspace=0.3)  # more vertical spacing and horizontal spacing
    plt.tight_layout(rect=[0, 0, 1, 0.95])      # leave space at top for suptitle & legend
    sns.despine(offset=15, trim=False)

    # --- Save figure ---
    if save:
        filename_base = f"{gas_label.lower()}_emissions_comparison"
        pdf_path = output_dir / f"{filename_base}.pdf"
        svg_path = output_dir / f"{filename_base}.svg"
        fig.savefig(pdf_path, format="pdf", bbox_inches="tight", dpi=300)
        fig.savefig(svg_path, format="svg", bbox_inches="tight")
        print(f"✅ Saved: {pdf_path}\n✅ Saved: {svg_path}")

    plt.show()


# --- Plot CO₂ figure ---
plot_gas(df_co2, gas_label="CO₂")

# --- Plot CH₄ figure ---
plot_gas(df_ch4, gas_label="CH₄")


## Emission profiles

In [ ]:
df_prof = pd.read_csv("emission_profile_comparison.csv", sep=";")
df_prof["Reservoir_name"] = df_prof["res"].map(reservoir_map)
df_prof.drop(columns=df_prof.columns[0], inplace=True)

def plot_profiles(df = df_prof, save=True):
    """ """
    reservoirs = df["Reservoir_name"].unique()
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()

    legend_handles = None
    legend_labels = None

    # --- Adjust spacing ---
    plt.subplots_adjust(hspace=0.9, wspace=0.3)  # more vertical spacing and horizontal spacing
    plt.tight_layout(rect=[0, 0, 1, 0.95])      # leave space at top for suptitle & legend
    sns.despine(offset=15, trim=False)

    for ax, reservoir in zip(axes, reservoirs):
        subset = df[df["Reservoir_name"] == reservoir]
        melt_df = subset.melt(
            id_vars=["year"], 
            value_vars=[
                "co2_profile_reemission",
                "ch4_profile_reemission",
                "total_reemission",
                "total_gres"
            ],
            var_name="Profile",
            value_name="Value"
        )

        model_labels = {
            "co2_profile_reemission": "CO₂ - RE-Emission",
            "ch4_profile_reemission": "CH₄ - RE-Emission",
            "total_reemission": "Total emission - RE-Emission",
            "total_gres": "Total emission - G-res Tool"
        }
        melt_df["Profile"] = melt_df["Profile"].map(model_labels)

        # --- Define special style overrides ---
        dashes = {
            "CO₂ - RE-Emission": "",
            "CH₄ - RE-Emission": "",
            "Total emission - RE-Emission": "",
            "Total emission - G-res Tool": (4, 2),  # Only this one dashed
        }
        markers = {
            "CO₂ - RE-Emission": "o",          # circle
            "CH₄ - RE-Emission": "o",           # circle
            "Total emission - RE-Emission": "o",  # circle
            "Total emission - G-res Tool": "X",   # cross
        }

        # Plot line first (so it appears beneath scatter points)
        sns.lineplot(
            data=melt_df[melt_df["Profile"] != "Total emission - G-res Tool"],
            x="year",
            y="Value",
            hue="Profile",
            palette=colors_profile,
            style="Profile",
            dashes=dashes,
            ax=ax,
            linewidth=1.0,
            alpha=0.7,
            legend=False,  # keep legend from scatter if you want consistent markers
        )

        sns.lineplot(
            data=melt_df[melt_df["Profile"] == "Total emission - G-res Tool"],
            x="year",
            y="Value",
            #hue="Profile",
            palette=colors_profile[-1:],
            style="Profile",
            dashes=dashes,
            ax=ax,
            linewidth=1.0,
            color="k",
            alpha=0.7,
            legend=False,  # keep legend from scatter if you want consistent markers
        )
        
        # Circles first
        sns.scatterplot(
            data=melt_df[melt_df["Profile"] != "Total emission - G-res Tool"],
            x="year",
            y="Value",
            hue="Profile",
            style="Profile",
            markers={"CO₂ - RE-Emission": "o", "CH₄ - RE-Emission": "o", "Total emission - RE-Emission": "o"},
            palette=colors_profile,
            ax=ax,
            s=50,            # normal size for circles
            edgecolor="k",
            alpha=0.8,
            linewidth=0.3,
        )
        
        # Cross separately, larger size
        sns.scatterplot(
            data=melt_df[melt_df["Profile"] == "Total emission - G-res Tool"],
            x="year",
            y="Value",
            #hue="Profile",
            style="Profile",
            markers={"Total emission - G-res Tool": "X"},
            palette=colors_profile[-1:],
            ax=ax,
            s=140,           # larger size for cross
            edgecolor="k",
            facecolor="none", # 'X' ignores this but safe for other markers
            alpha=1,
            linewidth=0.5,
        )

        # --- Titles & labels ---
        ax.set_title(reservoir, fontsize=14, fontweight="bold", pad=8)
        ax.set_xlabel("Year", fontsize=13)
        # y-label with two lines (main label + unit)
        ax.set_ylabel(f"Emission flux\n(gCO$_{{2e}}$ m$^{{-2}}$ yr$^{{-1}}$)", fontsize=13)
        #ax.set_ylabel(rf"{gas_label} flux\n(gCO$_{{2e}}$ m$^{{-2}}$ yr$^{{-1}}$)", fontsize=12)
        ax.tick_params(axis='x', rotation=0, labelsize=12)
        ax.tick_params(axis='y', labelsize=12)

        # Axis lines and grid
        for spine in ax.spines.values():
            spine.set_color("k")
            spine.set_linewidth(1.0)
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.grid(True, axis='y', linestyle='-', linewidth=0.35, alpha=0.8, color='0.7')

        # Capture handles and labels for shared legend
        if legend_handles is None:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()  # Remove per-panel legend

        # --- Shared legend below suptitle ---
        fig.suptitle(f"GHG net emission flux profiles: RE-Emission vs. G-res Tool", fontsize=16, fontweight="bold", y=0.98)
        fig.legend(
            legend_handles, legend_labels,
            loc="upper center", bbox_to_anchor=(0.5, 0.95),
            ncol=4,
            frameon=False, fontsize=12,
            handlelength=1.8, handleheight=1.5
        )
    
        # --- Adjust spacing ---
        plt.subplots_adjust(hspace=0.9, wspace=0.3)  # more vertical spacing and horizontal spacing
        plt.tight_layout(rect=[0, 0, 1, 0.95])      # leave space at top for suptitle & legend
        sns.despine(offset=15, trim=False)
    
    # --- Save figure ---
    if save:
        filename_base = f"emissions_profile_comparison"
        pdf_path = output_dir / f"{filename_base}.pdf"
        svg_path = output_dir / f"{filename_base}.svg"
        fig.savefig(pdf_path, format="pdf", bbox_inches="tight", dpi=300)
        fig.savefig(svg_path, format="svg", bbox_inches="tight")
        print(f"✅ Saved: {pdf_path}\n✅ Saved: {svg_path}")

    plt.show()
    
plot_profiles(save=True)